<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 38 · Tổng quan NLP và tiền xử lý tiếng Việt</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Mở đầu Phần IV: tải bộ phản hồi sinh viên UIT-VSFC, chuẩn hóa Unicode và vị trí dấu thanh, tách từ bằng underthesea, đếm từ vựng, thấy luật Zipf và cái giá của việc bỏ stopword.</p>
</div>

### Sau notebook này bạn sẽ

- Thấy vì sao hai chữ trông giống hệt mà máy coi là khác, và chuẩn hóa NFC, NFKC, kiểu đặt dấu.
- Tách từ tiếng Việt bằng underthesea, so âm tiết với từ.
- Tìm từ ghép chỉ bằng đếm (PMI).
- Đo luật Zipf, tỷ lệ từ ngoài từ vựng, và tác hại của bỏ stopword trên dữ liệu thật.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> khoảng 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch38-tong-quan-nlp/01-nlp-la-gi/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · NLP là gì](#trang-1)
2. [Trang 2 · Unicode và chuẩn hóa](#trang-2)
3. [Trang 3 · Tách từ tiếng Việt](#trang-3)
4. [Trang 4 · Từ vựng và luật Zipf](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · NLP là gì

Tải UIT-VSFC và nhìn dữ liệu.

*Đọc kèm trên web: [Trang 1 · NLP là gì](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch38-tong-quan-nlp/01-nlp-la-gi/)*

In [ ]:
!pip install -q underthesea                                 # thư viện xử lý tiếng Việt

In [ ]:
import os                                                   # đường dẫn
import re                                                   # biểu thức chính quy
import unicodedata                                          # chuẩn hóa Unicode
import urllib.request                                       # tải tệp
from collections import Counter                             # đếm
import numpy as np                                          # mảng
import pandas as pd                                         # bảng
URL = "https://huggingface.co/api/datasets/uitnlp/vietnamese_students_feedback/parquet/default/{}/0.parquet"   # tệp parquet trên Hugging Face
THU_MUC = os.path.expanduser("~/.cache/vsfc")               # lưu tạm
def tai(phan):                                              # tải một lần, lần sau đọc từ ổ đĩa
    os.makedirs(THU_MUC, exist_ok=True)                     # tạo thư mục
    tep = os.path.join(THU_MUC, phan + ".parquet")          # đường dẫn tệp
    if not os.path.exists(tep):                             # chưa có thì tải
        urllib.request.urlretrieve(URL.format(phan), tep)   # tải về
    return pd.read_parquet(tep)                             # đọc thành DataFrame
tr, va, te = tai("train"), tai("validation"), tai("test")   # ba phần chia sẵn của tác giả
print(len(tr), len(va), len(te), list(tr.columns))          # cỡ và các cột
print(tr.sentiment.value_counts().sort_index().tolist(), tr.topic.value_counts().sort_index().tolist())   # số câu theo nhãn

Output mong đợi:

```text
11426 1583 3166 ['sentence', 'sentiment', 'topic']
[5325, 458, 5643] [8166, 2201, 497, 562]
```

In [ ]:
for i in (0, 3, 6, 7):                                      # bốn câu
    print(tr.sentiment[i], tr.topic[i], tr.sentence[i])     # cảm xúc, chủ đề, câu

Output mong đợi:

```text
2 1 slide giáo trình đầy đủ .
0 0 chưa áp dụng công nghệ thông tin và các thiết bị hỗ trợ cho việc giảng dạy .
1 3 em sẽ nợ môn này , nhưng em sẽ học lại ở các học kỳ kế tiếp .
0 1 thời lượng học quá dài , không đảm bảo tiếp thu hiệu quả .
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Dữ liệu đã qua xử lý một phần: chữ thường, dấu câu tách riêng, tên giảng viên thay bằng mã như wzjwz208.</p>
</div>

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Unicode và chuẩn hóa

NFC, NFKC và vị trí dấu thanh.

*Đọc kèm trên web: [Trang 2 · Unicode và chuẩn hóa](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch38-tong-quan-nlp/02-unicode-va-chuan-hoa/)*

In [ ]:
a = "Hòa"                                                   # dạng dựng sẵn
b = unicodedata.normalize("NFD", a)                         # tách thành chữ gốc cộng dấu tổ hợp
print(a == b, len(a), len(b), [hex(ord(c)) for c in b])     # khác nhau dù trông giống hệt
print(a == unicodedata.normalize("NFC", b))                 # gộp lại thì bằng

Output mong đợi:

```text
False 3 4 ['0x48', '0x6f', '0x300', '0x61']
True
```

In [ ]:
print("Đà Nẵng".lower(), "ＡＩ２０２６".lower(), unicodedata.normalize("NFKC", "ＡＩ２０２６"))   # chữ thường và ký tự toàn khổ

Output mong đợi:

```text
đà nẵng ａｉ２０２６ AI2026
```

In [ ]:
print("hoà" == "hòa", unicodedata.normalize("NFC", "hoà") == "hòa")   # NFC không sửa được vị trí dấu

Output mong đợi:

```text
False False
```

In [ ]:
CU_MOI = {"oà": "òa", "oá": "óa", "oả": "ỏa", "oã": "õa", "oạ": "ọa",   # vần oa: kiểu cũ sang kiểu mới
          "oè": "òe", "oé": "óe", "oẻ": "ỏe", "oẽ": "õe", "oẹ": "ọe",   # vần oe
          "uỳ": "ùy", "uý": "úy", "uỷ": "ủy", "uỹ": "ũy", "uỵ": "ụy"}   # vần uy; chỉ đổi khi không có âm cuối
mau = re.compile(r"(?<!q)(" + "|".join(CU_MOI) + r")(?!\w)")   # bỏ qua sau q; không có chữ cái đứng sau
def dau_moi(s):                                             # đổi kiểu đặt dấu cũ sang mới
    return mau.sub(lambda m: CU_MOI[m.group(1)], s)         # thay từng chỗ khớp
print([dau_moi(w) for w in ["hoà", "thuý", "quý", "khoẻ", "hoàng", "hoà bình", "loè loẹt", "Quỳnh"]])   # thử các trường hợp

Output mong đợi:

```text
['hòa', 'thúy', 'quý', 'khỏe', 'hoàng', 'hòa bình', 'lòe loẹt', 'Quỳnh']
```

In [ ]:
tat_ca = pd.concat([tr, va, te]).sentence                   # mọi câu
cu = tat_ca[tat_ca.map(lambda s: dau_moi(s) != s)]          # câu có kiểu dấu cũ
khac = Counter(w for s in cu for w in s.split() if dau_moi(w) != w)   # các chữ kiểu cũ
print(len(cu), khac.most_common(5))                         # số câu, chữ hay gặp
print(Counter(w for s in tat_ca for w in s.split() if w in {dau_moi(x) for x in khac}).most_common(5))   # cùng chữ viết kiểu mới

Output mong đợi:

```text
39 [('hoà', 15), ('khoá', 14), ('hoạ', 8), ('hoá', 4), ('thoả', 1)]
[('hòa', 94), ('họa', 52), ('khóa', 48), ('hóa', 15), ('thỏa', 6)]
```

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Tách từ tiếng Việt

Tách từ bằng underthesea và tìm từ ghép bằng PMI.

*Đọc kèm trên web: [Trang 3 · Tách từ tiếng Việt](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch38-tong-quan-nlp/03-tach-tu/)*

In [ ]:
from underthesea import word_tokenize                       # tách từ tiếng Việt
print(word_tokenize("Bác sĩ khám bệnh cho bệnh nhân"))      # từ ghép
print(word_tokenize("Ông già đi nhanh quá"))                # câu mơ hồ
print(word_tokenize("học sinh học sinh học", format="text"))   # nối âm tiết bằng gạch dưới

Output mong đợi:

```text
['Bác sĩ', 'khám', 'bệnh', 'cho', 'bệnh nhân']
['Ông già', 'đi', 'nhanh', 'quá']
học_sinh học_sinh học
```

In [ ]:
import time                                                 # đo thời gian
t0 = time.time()                                            # bắt đầu
tu_tr = [word_tokenize(c) for c in tr.sentence]             # tách từ cả tập train
giay = time.time() - t0                                     # số giây
am_tiet_tr = [c.split() for c in tr.sentence]               # cắt theo dấu cách: âm tiết
n_am_tiet, n_tu = sum(map(len, am_tiet_tr)), sum(map(len, tu_tr))   # số token hai mức
dem_am_tiet = Counter(x for c in am_tiet_tr for x in c)     # từ vựng âm tiết
dem_tu = Counter(x for c in tu_tr for x in c)               # từ vựng từ
print(n_am_tiet, n_tu, round(n_am_tiet / n_tu, 3), len(dem_am_tiet), len(dem_tu), giay < 30)   # so hai mức
print([w for w, _ in dem_tu.most_common(40) if " " in w])   # từ ghép trong 40 từ đầu

Output mong đợi:

```text
163492 127413 1.283 2514 4009 True
['sinh viên', 'nhiệt tình', 'giảng viên', 'bài tập', 'giảng dạy', 'kiến thức', 'thực hành', 'môn học']
```

In [ ]:
import math                                                 # log
d1 = Counter(x for c in am_tiet_tr for x in c)              # đếm âm tiết
d2 = Counter((a, b) for c in am_tiet_tr for a, b in zip(c, c[1:]))   # đếm cặp liền nhau
N1, N2 = sum(d1.values()), sum(d2.values())                 # tổng
def pmi(a, b):                                              # cặp dính nhau hơn mức tình cờ bao nhiêu
    return math.log2((d2[(a, b)] / N2) / ((d1[a] / N1) * (d1[b] / N1)))   # log2 P(ab) / P(a)P(b)
cap = sorted(((round(pmi(a, b), 2), a + " " + b) for (a, b), n in d2.items()
              if n >= 20 and a.isalpha() and b.isalpha()), reverse=True)   # chỉ cặp gặp ít nhất 20 lần
print(cap[:8])                                              # tám cặp đầu
top30 = [w for _, w in cap[:30]]                            # 30 cặp đầu
print(sum(w in dem_tu for w in top30), [w for w in top30 if w not in dem_tu])   # trùng với từ của underthesea

Output mong đợi:

```text
[(12.72, 'tinh thần'), (12.67, 'khô khan'), (12.61, 'đơn giản'), (12.57, 'sẵn sàng'), (12.38, 'mơ hồ'), (12.38, 'giới thiệu'), (12.28, 'ảnh hưởng'), (12.27, 'trách nhiệm')]
29 ['sở vật']
```

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Từ vựng và luật Zipf

Luật Zipf, tốc độ tăng từ vựng, OOV và stopword.

*Đọc kèm trên web: [Trang 4 · Từ vựng và luật Zipf](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch38-tong-quan-nlp/04-tu-vung-va-zipf/)*

In [ ]:
print(dem_tu.most_common(12))                               # 12 từ thường gặp nhất
f = np.array(sorted(dem_tu.values(), reverse=True))         # tần suất theo hạng
print(round(f[:10].sum() / f.sum(), 3), round(f[:100].sum() / f.sum(), 3), int((f == 1).sum()))   # phần của 10, 100 từ đầu; số từ gặp 1 lần

Output mong đợi:

```text
[('.', 11009), (',', 6836), ('thầy', 3039), ('sinh viên', 2697), ('dạy', 2155), ('và', 2068), ('không', 2033), ('rất', 1961), ('nhiệt tình', 1839), ('cho', 1713), ('có', 1695), ('hiểu', 1654)]
0.277 0.677 1932
```

In [ ]:
hang = np.arange(1, len(f) + 1)                             # hạng 1, 2, 3...
giua = (hang >= 10) & (hang <= 1000)                        # bỏ hai đầu
print(np.polyfit(np.log(hang[giua]), np.log(f[giua]), 1).round(3))   # độ dốc trên trục log-log

Output mong đợi:

```text
[-1.387 11.744]
```

In [ ]:
da_gap, n, diem = set(), 0, []                              # từ đã gặp, số token, các điểm
for c in tu_tr:                                             # đi dần qua tập train
    for x in c:                                             # mỗi token
        n += 1                                              # đếm token
        da_gap.add(x)                                       # thêm vào từ vựng
    diem.append((n, len(da_gap)))                           # ghi lại sau mỗi câu
print([diem[k - 1] for k in (1000, 3000, 6000, 11426)])     # (số token, số từ khác nhau)

Output mong đợi:

```text
[(11075, 1236), (33107, 2079), (66621, 2896), (127413, 4009)]
```

In [ ]:
tu_te = [word_tokenize(c) for c in te.sentence]             # tách từ tập test
am_tiet_te = [c.split() for c in te.sentence]               # âm tiết tập test
oov = lambda ds, tu_vung: sum(x not in tu_vung for c in ds for x in c) / sum(map(len, ds))   # tỷ lệ token không có trong từ vựng
print(round(oov(am_tiet_te, dem_am_tiet), 4), round(oov(tu_te, dem_tu), 4))   # âm tiết, từ

Output mong đợi:

```text
0.0065 0.0156
```

In [ ]:
tieu_cuc, tich_cuc = tr[tr.sentiment == 0].sentence, tr[tr.sentiment == 2].sentence   # hai nhóm cảm xúc
print(round(tieu_cuc.str.contains(r"\bkhông\b").mean(), 3), round(tich_cuc.str.contains(r"\bkhông\b").mean(), 3))   # tỷ lệ câu có "không"

Output mong đợi:

```text
0.28 0.041
```

<div style="border-left:6px solid #C0392B;background-color:#FBECEA;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#962D22;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">BỎ STOPWORD</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">"Không" có trong 28% câu tiêu cực và 4,1% câu tích cực. Danh sách stopword có sẵn thường có từ này: bỏ đi là bỏ đúng tín hiệu cần nhất.</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Đoán trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch38-tong-quan-nlp/05-bai-tap/)*

### Bài 38.1 · Một hàm làm sạch

NFKC, chữ thường, đặt dấu kiểu mới, tách dấu câu, gộp khoảng trắng; thử trên một câu bẩn.
<details><summary>Xem đáp án</summary>

68 ký tự thành 58; hoà, khoẻ về kiểu mới, số toàn khổ về số thường. Thứ tự các bước là then chốt. Code ở ô ngay dưới.
</details>

In [ ]:
def lam_sach(s):                                            # làm sạch theo đúng thứ tự
    s = unicodedata.normalize("NFKC", s)                    # 1. dạng chuẩn, kể cả ký tự toàn khổ
    s = s.lower()                                           # 2. chữ thường
    s = dau_moi(s)                                          # 3. đặt dấu kiểu mới
    s = re.sub(r"([.,!?;:()])", r" \1 ", s)                # 4. tách dấu câu
    return re.sub(r"\s+", " ", s).strip()                  # 5. gộp khoảng trắng
ban = unicodedata.normalize("NFD", "Thầy HOÀ dạy môn Toán ２ rất hay,dễ hiểu!!") + "  Khoẻ   không?"   # câu bẩn cố ý
print(len(ban), [lam_sach(ban)], len(lam_sach(ban)))        # trước và sau
print(lam_sach(ban).split() == word_tokenize(lam_sach(ban)))   # âm tiết có trùng với từ không

Output mong đợi:

```text
68 ['thầy hòa dạy môn toán 2 rất hay , dễ hiểu ! ! khỏe không ?'] 58
False
```

### Bài 38.2 · Chọn cỡ từ vựng

Giữ từ gặp ít nhất k lần, với k = 1, 2, 5.
<details><summary>Xem đáp án</summary>

k = 2 cắt từ vựng gần một nửa (4009 xuống 2077) mà OOV chỉ tăng từ 1,56% lên 2,29%. Code ở ô ngay dưới.
</details>

In [ ]:
for k in (1, 2, 5):                                         # ba ngưỡng
    tu_vung = {w for w, n in dem_tu.items() if n >= k}      # từ đủ thường
    print(k, len(tu_vung), round(oov(tu_te, tu_vung), 4))   # cỡ từ vựng, OOV trên test

Output mong đợi:

```text
1 4009 0.0156
2 2077 0.0229
5 1217 0.0371
```

### Bài 38.3 · Cái giá của stopword

Bỏ 15 stopword rồi đếm câu thành giống hệt một câu mang nhãn cảm xúc ngược lại.
<details><summary>Xem đáp án</summary>

Không bỏ: 1 câu (lỗi nhãn sẵn có). Bỏ 15 stopword: 20 câu, vì 'không' đã bị xóa. Code ở ô ngay dưới.
</details>

In [ ]:
from collections import defaultdict                         # dict có giá trị mặc định
STOP = {"và", "của", "thì", "là", "không", "rất", "cho", "có", "được", "các", "những", "với", "này", ".", ","}   # danh sách hay gặp
def xung_dot(bo):                                           # câu giống hệt nhau mà có cả nhãn 0 và 2
    nhom = defaultdict(set)                                 # câu sau khi bỏ, tập nhãn
    for c, y in zip(tr.sentence, tr.sentiment):             # mọi câu train
        nhom[" ".join(w for w in word_tokenize(c) if w not in bo)].add(y)   # bỏ stopword rồi gom
    return [k for k, v in nhom.items() if {0, 2} <= v]      # có cả tiêu cực lẫn tích cực
print(len(xung_dot(set())), len(xung_dot(STOP)), xung_dot(STOP)[:6])   # không bỏ, có bỏ

Output mong đợi:

```text
1 20 ['nhiệt tình sinh viên', 'thầy vui tính', 'thầy dạy hiệu quả', 'nhàm chán', 'thầy dạy tâm', 'giảng viên nhiệt tình tận tâm']
```